# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/J-Aamir/Capstone-Flyrank/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
# Setup: clone repo (for consistency with other notebooks) + install HF datasets lib + auth
import os
if not os.path.exists("Capstone-Flyrank"):
    !git clone https://github.com/J-Aamir/Capstone-Flyrank.git
%cd Capstone-Flyrank
!pip install -q huggingface_hub duckdb pandas

from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")  # set this in Colab Secrets first


## 1. Unit of analysis + time window

**One row = one content page, on one report date, for one client** (`report_date x client_id x content_id`, from `fact_content_daily_performance`). **Time window: `month=2026-03`** (mid-panel, not the `_sample` June-2026 sealed test month, per the warehouse warning). Verified with queries below.

In [ ]:
import duckdb, os

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute("CREATE SECRET hf_token (TYPE HUGGINGFACE, TOKEN '" + os.environ["HF_TOKEN"] + "');")

# Point at the mid-panel month partition only (not the full 79M-row table, not _sample)
MONTH = "2026-03"
path = f"hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month={MONTH}/*.parquet"
print("Target partition:", path)

# Sanity ping — list a couple rows before running full checks
preview = con.execute(f"SELECT * FROM read_parquet('{path}') LIMIT 3").df()
preview


## 2. Fields: feature / label / context / excluded

| Field | Bucket | Why |
|---|---|---|
| `gsc_clicks`, `gsc_impressions`, `gsc_avg_position` (prior window) | feature | knowable before the decision moment |
| `ga4_sessions`, `ga4_engaged_sessions` (prior window, `ga4_data_available IS TRUE`) | feature | knowable before, but must filter the availability flag |
| a future-window impressions drop (built from a *later* month, not in this slice) | label | the outcome I'd predict, observed after the decision point |
| `content_id`, `client_id`, `report_date` | context | identifiers for grouping/joining, never model inputs |
| `gsc_avg_position` **from the same window as the label** | excluded | using same-period position to explain same-period decline is circular, not predictive |


In [ ]:
# Fields exist only as this markdown table for now — verified with real queries in Section 3.
print("Feature/label/context/excluded table defined above.")


## 3. Verify it with queries (grain, counts, missing values, windows)

Three required checks: (a) grain — one row really is report_date x client_id x content_id, (b) row count + date span for this slice, (c) availability — filter `ga4_data_available IS TRUE` and show the survival count.

In [ ]:
# (a) Grain check — should return ZERO rows if the grain is really one-row-per-key
grain_check = con.execute(f"""
    SELECT report_date, client_id, content_id, COUNT(*) c
    FROM read_parquet('{path}')
    GROUP BY 1,2,3
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
print("Grain violations (should be empty):")
print(grain_check)

# (b) Row count + date span
span = con.execute(f"""
    SELECT COUNT(*) AS n_rows, MIN(report_date) AS min_date, MAX(report_date) AS max_date
    FROM read_parquet('{path}')
""").df()
print("Row count + date span:")
print(span)

# (c) Availability filter
avail = con.execute(f"""
    SELECT COUNT(*) AS n_total,
           SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS n_available
    FROM read_parquet('{path}')
""").df()
print("Availability check (GA4):")
print(avail)


## 4. Data limits

- **Unbalanced history:** clients differ wildly in how much history they have (`dim_clients.gsc_data_start`), so a single global date window over- or under-represents some clients.
- **GA4-only-partial rows:** rows before a client's `ga4_data_start` are zero-filled with `ga4_data_available = FALSE` — those zeros mean "no data," not "no engagement," so they must be filtered, not treated as real zeros.
- **Window overlap risk:** the query table's 90-day window can overlap the label's outcome window, so any label built from a later period must use only `*_prev30`-style features from before it, or the model can see into its own future.


In [ ]:
# --- Five features (max), each with an "available when?" line ---
# 1. gsc_impressions_prior30  -- knowable at the decision moment: it's already-observed traffic before the label window
# 2. gsc_avg_position_prior30 -- knowable before: prior-window rank, not the label-window rank
# 3. ga4_engagement_rate_prior30 (filtered ga4_data_available IS TRUE) -- knowable before, once availability is checked
# 4. content_age_days -- knowable at any time, it's just today's date minus publish date
# 5. days_since_last_update -- knowable at any time, a static/slow-changing fact about the page

# --- The trap: add ONE label-derived column on purpose ---
# e.g. add "same_window_position_change" (built FROM the label window itself) as a 6th feature,
# retrain/rescore, watch precision jump toward ~1.0, then delete it and report the honest number.
# (Exact code depends on your label-building query — fill in once Section 3 output confirms the grain.)
print("Feature list defined above; leakage trap to be run once queries in Section 3 succeed.")


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.